In [2]:
%pip install anthropic python-dotenv

     ------------------------------------- 662.1/662.1 kB 13.9 MB/s eta 0:00:00
     ---------------------------------------- 73.5/73.5 kB ? eta 0:00:00
     ---------------------------------------- 204.7/204.7 kB ? eta 0:00:00
     ------------------------------------- 472.0/472.0 kB 14.9 MB/s eta 0:00:00
     ---------------------------------------- 44.6/44.6 kB ? eta 0:00:00
     ---------------------------------------- 78.8/78.8 kB ? eta 0:00:00
     ---------------------------------------- 2.1/2.1 MB 32.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.9.0
    Uninstalling typing_extensions-4.9.0:
      Successfully uninstalled typing_extensions-4.9.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires ml-dtypes~=0.2.0, but you have ml-dtypes 0.3.2 which is incompatible.

[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#Loan Env Variable
import json
from dotenv import load_dotenv

load_dotenv()



True

In [4]:
# Create an API Client 
from anthropic import Anthropic

client = Anthropic()
model = "claude-opus-4-5-20251101"

In [5]:
# Make a request

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, stop_sequences=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        # "temperature" : temperature,
    }
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences


    message = client.messages.create(**params)
    return message.content[0].text


In [6]:
#Make a starting list of messages
messages = []

#Add in the initailise user quastion of "Define Quantum Computing in one sentence"
add_user_message(messages, "Define Quantum Computing in one sentence")

# Pass the list of the message into 'chat' to get an answer
answer = chat(messages)

#take the answer and add it as assistant message into our list 
add_assistant_message(messages, answer)

#Add in the user's follow-up quastion 
add_user_message(messages, "Write another sentence")

answer = chat(messages)
# answer

In [7]:
# # Chat Bot Loop

# # Make an Initail list of messages
# messages = []

# #use a 'while True' loop to run the chatbot forever
# while True:
#     #get user input
#     user_input = input("> ")
#     print(">", user_input)

#     #add the user input as a message to the list of messages
#     add_user_message(messages, user_input)
#     #call calude with the 'chat' funtion
#     answer = chat(messages)
#     #Add generated text to the list of messages
#     add_assistant_message(messages, answer)
#     #print the generated text
#     print("-----")
#     print(answer)
#     print("-----")

In [ ]:
messages = []

# system = """
#     You are a pateint math tutor. 
#     Do not directly answer a student's quastion.
#     Guide them to a solution step by step 
# """

add_user_message(messages, "Gemerate a One sentance movie idea")
# answer = chat(messages)
# answer = chat(messages, system = "You are a Python engineer who write very concise code.")
answer = chat(messages, temperature=1.0)    

answer

In [ ]:
messages = []

add_user_message(messages, "Write a 1 description of a fake database ")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

In [79]:
messages = []

add_user_message(messages, "Write a 1 description of a fake database ")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass
stream.get_final_message()

ParsedMessage(id='msg_01QoFCpyv6TvuX2k13hqfLDR', container=None, content=[ParsedTextBlock(citations=None, text='# **NexusVault DB**\n\nNexusVault is a cutting-edge, AI-powered quantum-hybrid database management system developed by the fictional company **Solaris Data Corp** (est. 2031). Unlike traditional relational or NoSQL databases, NexusVault uses a proprietary storage architecture called **"Dimensional Sharding"**, which organizes data across multiple virtual "layers" that mimic how the human brain categorizes information — by context, emotion, and relevance rather than simple rows and tables. It supports a custom query language called **NexQL** (pronounced "nex-quel"), which allows users to write queries in near-natural language, such as `FIND ALL customers WHO feel frustrated WITHIN the last 7 cycles`. NexusVault boasts zero-latency reads, self-healing data nodes, and built-in "memory decay" — a feature that automatically deprioritizes and archives data that hasn\'t been accesse

In [12]:
def generate_dataset():
    prompt = """Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python|json|regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects. """


    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)